# Online Retail — Customer Analytics & RFM Segmentation
### Capstone Project

**Dataset:** UCI Online Retail (id = 352) — every transaction of a UK-based, non-store online gift retailer between **01-Dec-2010 and 09-Dec-2011**. Many of its customers are wholesalers.

**Notebook roadmap** (follows the project guideline):

| # | Section | # | Section |
|---|---|---|---|
| 1 | Business Understanding | 9 | RFM Analysis |
| 2 | Dataset Loading & Documentation | 10 | RFM Visualization & Interpretation |
| 3 | Data Audit | 11 | Preprocessing for Clustering |
| 4 | Cleaning & Data Quality Decisions | 12 | Clustering & Evaluation |
| 5 | Feature Engineering | 13 | Segment Profiling |
| 6 | EDA — Sales / Time / Products / Customers / Geography / Baskets / Returns | 14 | Business Recommendations |
| 7 | Business Insights (10+) | 15 | Limitations & Next Steps |
| 8 | Customer-Level Feature Engineering | 16 | Final Conclusion |

> Run the cells top to bottom. Every number quoted in the text is printed by a code cell directly above it, so the analysis is reproducible.

## 1. Business Understanding

**The business.** A UK online retailer selling all-occasion giftware. A large share of buyers are small shops and wholesalers buying in trade quantities, which makes the customer base very uneven: a few accounts generate most of the revenue.

**Why this analysis.** Marketing currently treats every customer the same. That wastes discounts on loyal buyers who would buy anyway and misses customers who are quietly drifting away. The goal is to turn one year of transactions into **segments the business can act on**.

**Questions the business needs answered *before* any clustering:**

1. Which products generate the most units sold and the most revenue?
2. Which customers contribute the most revenue and orders?
3. How does revenue change over time? Which months, weekdays and hours are strongest?
4. Which countries generate the most customers, orders and revenue?
5. Are there seasonal purchasing patterns?
6. How common are cancellations/returns and which products/customers drive them?
7. What does a typical customer's purchasing behaviour look like?
8. Is there evidence of wholesale / bulk-buying behaviour?
9. Can customers be grouped into actionable segments based on value and engagement?

**Unit of analysis — important.** One row in this dataset is a *transaction line* (one product on one invoice), **not** an order. Orders are counted as unique `InvoiceNo`, and customers as unique `CustomerID`. Mixing these up is one of the most common mistakes in this dataset.

## 2. Dataset Loading & Documentation

In [ ]:
# Install the UCI repository client (only needed once per environment)
%pip install -q ucimlrepo

In [ ]:
# ---- Imports & global settings -------------------------------------------------
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

warnings.filterwarnings('ignore')
RANDOM_STATE = 42                      # fixed seed -> reproducible clustering
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# Consistent, colour-blind-checked palette (fixed order, never cycled)
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
BLUE, ORANGE, AQUA, YELLOW = PALETTE[:4]
MUTED = '#898781'
plt.rcParams.update({
    'figure.figsize': (11, 4.5), 'figure.dpi': 100,
    'axes.facecolor': '#fcfcfb', 'figure.facecolor': 'white',
    'axes.edgecolor': '#e1e0d9', 'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'bold', 'axes.titlesize': 12, 'axes.labelcolor': '#52514e',
    'xtick.color': '#52514e', 'ytick.color': '#52514e', 'lines.linewidth': 2,
    'axes.prop_cycle': plt.cycler(color=PALETTE),
})

def gbp(x):
    # Format a number as pounds sterling
    return f'£{x:,.0f}'

def gbp_axis(ax, axis='y'):
    # Short £ labels on an axis (e.g. £1.2M, £350k)
    def f(v, _):
        if abs(v) >= 1e6: return f'£{v/1e6:.1f}M'
        if abs(v) >= 1e3: return f'£{v/1e3:.0f}k'
        return f'£{v:.0f}'
    (ax.yaxis if axis == 'y' else ax.xaxis).set_major_formatter(mtick.FuncFormatter(f))

OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Environment ready.')

In [ ]:
# ---- Fetch the dataset from the UCI repository ----------------------------------
from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)

# data (as pandas dataframes)
X = online_retail.data.features
y = online_retail.data.targets      # None: this is not a supervised-learning dataset

# metadata
print(online_retail.metadata)

# variable information
print(online_retail.variables)

In [ ]:
# ucimlrepo splits the columns into 'ids' (InvoiceNo, StockCode) and 'features'.
# We need all 8 original columns, so we use data.original (or rebuild it if absent).
COLS = ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

if getattr(online_retail.data, 'original', None) is not None:
    raw = online_retail.data.original.copy()
else:
    raw = pd.concat([online_retail.data.ids, online_retail.data.features], axis=1)

raw = raw[COLS].copy()
print(f'Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
raw.head()

### Data dictionary

| Column | Meaning | Type we will use | Notes |
|---|---|---|---|
| `InvoiceNo` | Invoice number, unique per order | string | Starts with **"C"** = cancellation / return |
| `StockCode` | Product code | string | Some codes are not products (`POST`, `M`, `D`, `BANK CHARGES`, …) |
| `Description` | Product name | string | Free text, some missing |
| `Quantity` | Units on this line | int | Negative on cancellations and stock adjustments |
| `InvoiceDate` | Date & time of the invoice | datetime | |
| `UnitPrice` | Price per unit in GBP (£) | float | 0 = free/adjustment, < 0 = bad-debt adjustment |
| `CustomerID` | Customer identifier | **string (categorical)** | **Never** used as a number. Missing for guest checkouts |
| `Country` | Customer's country | string | Heavily UK |

## 3. Data Audit

Before changing anything we measure every data-quality issue. Each finding becomes a documented decision in Section 4.

In [ ]:
audit = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing': raw.isna().sum(),
    'missing_%': (raw.isna().mean() * 100).round(2),
    'unique_values': raw.nunique(),
})
display(audit)
display(raw.describe().T)

In [ ]:
inv = raw['InvoiceNo'].astype(str).str.strip()
code = raw['StockCode'].astype(str).str.strip().str.upper()

checks = pd.Series({
    'Total rows (transaction lines)': len(raw),
    'Exact duplicate rows': raw.duplicated().sum(),
    'Rows with missing CustomerID': raw['CustomerID'].isna().sum(),
    'Rows with missing Description': raw['Description'].isna().sum(),
    'Cancellation lines (InvoiceNo starts with C)': inv.str.startswith('C').sum(),
    'Lines with Quantity < 0': (raw['Quantity'] < 0).sum(),
    'Quantity < 0 but NOT a cancellation invoice': ((raw['Quantity'] < 0) & ~inv.str.startswith('C')).sum(),
    'Lines with UnitPrice == 0': (raw['UnitPrice'] == 0).sum(),
    'Lines with UnitPrice < 0': (raw['UnitPrice'] < 0).sum(),
    'Unique invoices': inv.nunique(),
    'Unique customers': raw['CustomerID'].nunique(),
    'Unique stock codes': code.nunique(),
    'Countries': raw['Country'].nunique(),
}, name='count')
display(checks.to_frame().style.format('{:,}'))

In [ ]:
# Stock codes that contain no digits are usually NOT products (postage, fees, manual entries)
no_digit = (raw.assign(StockCode=code)
               .loc[~code.str.contains(r'\d'), ['StockCode', 'Description']]
               .groupby('StockCode')
               .agg(lines=('Description', 'size'),
                    example_description=('Description', lambda s: s.dropna().mode().iat[0] if s.notna().any() else None))
               .sort_values('lines', ascending=False))
no_digit

**Audit findings (to be resolved in Section 4)**

1. **Mixed types** — `InvoiceDate` is text, `CustomerID` is a float (must become a categorical ID), `InvoiceNo` mixes numbers and `C`-prefixed strings.
2. **Exact duplicates** exist — the same line captured twice would double-count revenue.
3. **About a quarter of lines have no `CustomerID`.** These cannot be attributed to a customer, but they are still real sales of real products.
4. **Cancellations** (`C` invoices) carry negative quantities. They are real refunds and must be *netted*, not silently deleted.
5. **Zero / negative prices and non-cancellation negative quantities** are stock adjustments, write-offs and giveaways, not customer sales.
6. **Non-product stock codes** (postage, Amazon fees, bank charges, manual adjustments, discounts) would distort product rankings.

## 4. Cleaning & Data Quality Decisions

Every step is logged with the number of rows affected, the decision taken and its **business impact**. Rows are *isolated with a flag* rather than deleted, so nothing is lost and every decision can be audited later.

In [ ]:
cleaning_log = []
def log_step(step, rows, decision, impact):
    cleaning_log.append({'step': step, 'rows_affected': int(rows), 'decision': decision, 'business_impact': impact})

df = raw.copy()

### 4.1 Structural cleansing & type casting

In [ ]:
# InvoiceNo / StockCode: text identifiers (strip spaces, remove any '.0' float artefact)
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df['StockCode'] = df['StockCode'].astype(str).str.strip().str.upper()
df['Description'] = df['Description'].astype('string').str.strip()
df['Country'] = df['Country'].astype(str).str.strip()

# InvoiceDate: text -> datetime (UCI format is month/day/year hour:minute)
if not pd.api.types.is_datetime64_any_dtype(df['InvoiceDate']):
    parsed = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M', errors='coerce')
    if parsed.isna().mean() > 0.01:              # fall back to automatic parsing if the format differs
        parsed = pd.to_datetime(df['InvoiceDate'], errors='coerce')
    df['InvoiceDate'] = parsed
assert df['InvoiceDate'].notna().all(), 'Some dates could not be parsed'

# CustomerID: an identifier, NOT a number -> nullable string (e.g. '17850')
df['CustomerID'] = df['CustomerID'].astype('Int64').astype('string')

log_step('Type casting', len(df), 'InvoiceDate -> datetime; CustomerID/InvoiceNo/StockCode -> string',
         'Enables time analysis; prevents CustomerID being treated as a numeric feature')
print(df.dtypes)
print('Date range:', df['InvoiceDate'].min(), '->', df['InvoiceDate'].max())

### 4.2 Exact duplicates

In [ ]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
log_step('Exact duplicate lines', n_dup, 'Removed (identical in all 8 columns)',
         'Prevents double counting of revenue and units')
print(f'Removed {n_dup:,} exact duplicate lines -> {len(df):,} rows remain')

### 4.3 Missing `CustomerID` — guest checkouts, not noise

We do **not** blindly drop these rows. First we group them by `InvoiceNo`: if an invoice is either *fully* missing or *fully* present, the missing IDs are whole guest-checkout orders (a system/channel effect), not random gaps inside an order.

In [ ]:
miss_share = df['CustomerID'].isna().groupby(df['InvoiceNo']).mean()
inv_status = pd.cut(miss_share, bins=[-0.01, 0, 0.999, 1], labels=['ID present on all lines', 'Partially missing', 'Missing on all lines (guest)'])
display(inv_status.value_counts().to_frame('invoices'))

df['Is_Guest'] = df['CustomerID'].isna()
guest = df[df['Is_Guest']]
reg = df[~df['Is_Guest']]

profile = pd.DataFrame({
    'lines': [len(reg), len(guest)],
    'invoices': [reg['InvoiceNo'].nunique(), guest['InvoiceNo'].nunique()],
    'lines_per_invoice': [len(reg) / reg['InvoiceNo'].nunique(), len(guest) / guest['InvoiceNo'].nunique()],
    'revenue_£': [(reg['Quantity'] * reg['UnitPrice']).sum(), (guest['Quantity'] * guest['UnitPrice']).sum()],
    'share_UK_%': [(reg['Country'] == 'United Kingdom').mean() * 100, (guest['Country'] == 'United Kingdom').mean() * 100],
    'share_price_0_%': [(reg['UnitPrice'] == 0).mean() * 100, (guest['UnitPrice'] == 0).mean() * 100],
}, index=['Registered customer', 'Guest (no CustomerID)'])
display(profile)

log_step('Missing CustomerID', df['Is_Guest'].sum(),
         'Kept & flagged Is_Guest. Used for product/time/country EDA; excluded from customer-level RFM',
         'Guest revenue is real sales; but it cannot be attributed to a customer, so RFM covers registered customers only')

**Decision.** Missing IDs come in whole invoices, so these are guest checkouts. Guest invoices are large (many lines each) and are a real revenue stream. They stay in the product, time and country analysis. They are excluded only from the customer-level work (RFM and clustering), where an ID is required.

### 4.4 Non-product stock codes

In [ ]:
NON_PRODUCT_CODES = {'POST', 'DOT', 'M', 'D', 'C2', 'S', 'B', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'PADS'}
df['Is_NonProduct'] = df['StockCode'].isin(NON_PRODUCT_CODES) | df['StockCode'].str.startswith('GIFT_')

np_summary = (df[df['Is_NonProduct']]
              .assign(Revenue=lambda d: d['Quantity'] * d['UnitPrice'])
              .groupby('StockCode').agg(lines=('Revenue', 'size'), revenue=('Revenue', 'sum'))
              .sort_values('lines', ascending=False))
display(np_summary)

log_step('Non-product stock codes', df['Is_NonProduct'].sum(),
         'Flagged Is_NonProduct (postage, fees, manual, discounts, vouchers); excluded from product & customer analysis',
         'Stops postage and fees appearing as "best-selling products" and keeps Monetary = merchandise spend')

### 4.5 Zero / negative prices and stock adjustments

Lines with `UnitPrice <= 0`, and negative quantities on *non*-cancellation invoices, are internal adjustments (damaged, lost, sample, bad-debt write-offs, giveaways). They are isolated, not deleted.

In [ ]:
is_cancel = df['InvoiceNo'].str.startswith('C')
df['Is_Anomaly'] = (df['UnitPrice'] <= 0) | ((df['Quantity'] < 0) & ~is_cancel)

anom = df[df['Is_Anomaly']]
print(f"Anomaly lines: {len(anom):,}  |  UnitPrice == 0: {(anom['UnitPrice'] == 0).sum():,}  |  UnitPrice < 0: {(anom['UnitPrice'] < 0).sum():,}")
print('Share of anomaly lines without a CustomerID: {:.1%}'.format(anom['Is_Guest'].mean()))
print('\nMost common descriptions on anomaly lines (shows they are internal adjustments):')
display(anom['Description'].fillna('<missing>').value_counts().head(12).to_frame('lines'))

log_step('Zero/negative price & stock adjustments', df['Is_Anomaly'].sum(),
         'Flagged Is_Anomaly and isolated from sales analysis',
         'Adjustments, write-offs and giveaways are not customer demand; keeping them would distort AOV and prices')

### 4.6 Cancellations — match to the original invoice and net out revenue

A cancellation line (`C…`) should cancel an earlier purchase of the **same product by the same customer** with at least that quantity. We try to find that original purchase. Cancellations that cannot be matched usually refer to purchases made *before* the dataset starts (before Dec-2010), or were split across several invoices.

**Decision:** all valid cancellation lines stay in the data as **negative revenue**, so every revenue figure is *net of returns*. Deleting them would overstate revenue and hide the return problem.

In [ ]:
df['IsCancelled'] = is_cancel
valid = ~df['Is_Anomaly'] & ~df['Is_NonProduct']

purchases = df[valid & ~df['IsCancelled'] & ~df['Is_Guest']]
cancels   = df[valid &  df['IsCancelled']]

# Candidate originals: same customer + same product, bought on/before the cancellation, with enough quantity
cand = (cancels[~cancels['Is_Guest']].reset_index()
        .merge(purchases.reset_index()[['index', 'CustomerID', 'StockCode', 'InvoiceDate', 'Quantity', 'InvoiceNo']],
               on=['CustomerID', 'StockCode'], suffixes=('_c', '_p')))
cand = cand[(cand['InvoiceDate_p'] <= cand['InvoiceDate_c']) & (cand['Quantity_p'] >= -cand['Quantity_c'])]
best = cand.sort_values('InvoiceDate_p').groupby('index_c').tail(1)   # most recent qualifying purchase

df['CancelMatched'] = False
df.loc[best['index_c'], 'CancelMatched'] = True
df['CancelledLater'] = False
df.loc[best['index_p'].unique(), 'CancelledLater'] = True   # purchase lines that were (partly) cancelled

c = df[valid & df['IsCancelled']]
c_val = -(c['Quantity'] * c['UnitPrice'])
print(f"Valid cancellation lines: {len(c):,} on {c['InvoiceNo'].nunique():,} cancellation invoices")
print(f"Matched to an original purchase: {c['CancelMatched'].mean():.1%} of lines, "
      f"{c_val[c['CancelMatched']].sum() / c_val.sum():.1%} of cancelled value")
print(f"Total value refunded: {gbp(c_val.sum())}")

log_step('Cancellations (C invoices)', len(c),
         'Kept as negative revenue; matched to original purchase where possible (CancelMatched flag)',
         'All revenue reported net of returns; return behaviour becomes an analysable signal')

In [ ]:
# The largest single cancellations — typically huge orders reversed within minutes (data-entry or test orders)
big = (c.assign(CancelValue=c_val)
        .nlargest(5, 'CancelValue')[['InvoiceNo', 'InvoiceDate', 'CustomerID', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'CancelValue', 'CancelMatched']])
big

These extreme lines show *why* netting matters: a handful of enormous orders were placed and cancelled almost immediately. Counted as gross sales they would make those products look like best-sellers and those customers look like whales. Netting removes the distortion automatically.

In [ ]:
log_df = pd.DataFrame(cleaning_log)
display(log_df)
log_df.to_csv(f'{OUT_DIR}/cleaning_log.csv', index=False)

## 5. Feature Engineering

Transaction-level features:

* **Time:** `Year`, `Month`, `YearMonth`, `DayOfWeek`, `Hour`, `Date` → seasonality and operational timing.
* **`TotalLineRevenue = Quantity × UnitPrice`** → negative on cancellation lines, so sums are automatically net.
* **`Is_UK`** → the data is heavily UK-skewed; this flag lets us look at international trends without the domestic market swamping them.

In [ ]:
DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

df['TotalLineRevenue'] = df['Quantity'] * df['UnitPrice']
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = pd.Categorical(df['InvoiceDate'].dt.day_name(), categories=DAY_ORDER, ordered=True)
df['Hour'] = df['InvoiceDate'].dt.hour
df['Date'] = df['InvoiceDate'].dt.normalize()
df['Is_UK'] = df['Country'].eq('United Kingdom')

# ---- Analysis-ready tables ------------------------------------------------------
tx      = df[~df['Is_Anomaly'] & ~df['Is_NonProduct']].copy()   # product lines, purchases + cancellations (NET)
sales   = tx[~tx['IsCancelled']].copy()                          # product purchases only (GROSS)
returns = tx[tx['IsCancelled']].copy()                           # product cancellations (negative revenue)

# Order-level table: one row per invoice (an ORDER), not per line
orders = (sales.groupby('InvoiceNo')
               .agg(InvoiceDate=('InvoiceDate', 'first'), CustomerID=('CustomerID', 'first'),
                    Country=('Country', 'first'), Is_UK=('Is_UK', 'first'), Is_Guest=('Is_Guest', 'first'),
                    Revenue=('TotalLineRevenue', 'sum'), Units=('Quantity', 'sum'),
                    Lines=('StockCode', 'size'), DistinctProducts=('StockCode', 'nunique')))

# One readable description per product (most frequent spelling)
desc_map = sales.groupby('StockCode')['Description'].agg(lambda s: s.dropna().mode().iat[0] if s.notna().any() else '')

print(f'tx (net product lines): {len(tx):,} | sales lines: {len(sales):,} | return lines: {len(returns):,} | orders: {len(orders):,}')
tx[['InvoiceNo', 'InvoiceDate', 'StockCode', 'Quantity', 'UnitPrice', 'TotalLineRevenue', 'YearMonth', 'DayOfWeek', 'Hour', 'Is_UK', 'IsCancelled']].head()

## 6. Exploratory Data Analysis

Each analysis answers one of the business questions from Section 1 and ends with an interpretation.

### 6.1 Sales overview

In [ ]:
gross = sales['TotalLineRevenue'].sum()
refunds = -returns['TotalLineRevenue'].sum()
net = tx['TotalLineRevenue'].sum()

kpis = pd.Series({
    'Period': f"{df['InvoiceDate'].min():%d %b %Y} – {df['InvoiceDate'].max():%d %b %Y}",
    'Gross product revenue': gbp(gross),
    'Refunded (cancellations)': f'{gbp(refunds)}  ({refunds / gross:.1%} of gross)',
    'Net product revenue': gbp(net),
    'Orders (unique invoices)': f'{len(orders):,}',
    'Registered customers who bought': f"{sales['CustomerID'].nunique():,}",
    'Guest-checkout share of revenue': f"{orders.loc[orders['Is_Guest'], 'Revenue'].sum() / gross:.1%}",
    'Products sold': f"{sales['StockCode'].nunique():,}",
    'Countries': f"{sales['Country'].nunique()}",
    'Average order value (AOV)': gbp(orders['Revenue'].mean()),
    'Median order value': gbp(orders['Revenue'].median()),
    'Median units per order': f"{orders['Units'].median():,.0f}",
})
kpis.to_frame('value')

**Interpretation.** The mean order value sits well above the median. That gap is the first sign of a **right-skewed, wholesale-driven business**: most orders are moderate, but a minority of very large trade orders pulls the average up. This skew will matter again when we scale features for clustering (Section 11).

### 6.2 Time — how does revenue change over time?

In [ ]:
monthly = (tx.groupby('YearMonth')
             .agg(NetRevenue=('TotalLineRevenue', 'sum'),
                  Orders=('InvoiceNo', lambda s: s[~s.str.startswith('C')].nunique()),
                  ActiveCustomers=('CustomerID', 'nunique'),
                  TradingDays=('Date', 'nunique')))
monthly['RevenuePerTradingDay'] = monthly['NetRevenue'] / monthly['TradingDays']
monthly['MoM_growth_%'] = monthly['NetRevenue'].pct_change() * 100
display(monthly)

last = monthly.index.max()
print(f'Note: {last} contains only {monthly.loc[last, "TradingDays"]} trading days (data ends {df["InvoiceDate"].max():%d %b %Y}) — it is a partial month.')

In [ ]:
# Small multiples instead of a dual axis: revenue, orders and active customers share the x-axis
x = monthly.index.astype(str)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
for ax, col, title in zip(axes, ['NetRevenue', 'Orders', 'ActiveCustomers'],
                          ['Net revenue per month', 'Orders per month', 'Active customers per month']):
    ax.plot(x, monthly[col], marker='o', color=BLUE, markersize=5)
    ax.plot(x[-1], monthly[col].iloc[-1], marker='o', color='white', markeredgecolor=BLUE, markersize=8)   # hollow = partial month
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=60)
gbp_axis(axes[0])
fig.text(0.5, -0.06, 'Hollow marker = partial month (data ends early December 2011)', ha='center', color=MUTED)
plt.tight_layout(); plt.show()

In [ ]:
# Fair comparison including the partial month: revenue per trading day
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x, monthly['RevenuePerTradingDay'], color=BLUE, width=0.7)
ax.set_title('Net revenue per trading day (adjusts for the partial last month)')
gbp_axis(ax); ax.tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

full_2011 = monthly.loc[(monthly.index.year == 2011) & (monthly.index != last)]
q4_share = full_2011.loc[full_2011.index.month >= 9, 'NetRevenue'].sum() / full_2011['NetRevenue'].sum()
print(f'Peak month: {monthly["NetRevenue"].idxmax()}  ({gbp(monthly["NetRevenue"].max())})')
print(f'Sep–Nov 2011 = {q4_share:.1%} of Jan–Nov 2011 net revenue (3 of 11 months = 27% if revenue were flat)')

**Interpretation — seasonality.** Revenue is fairly flat in the first half of 2011, then climbs from **September** and peaks in **November**: the Q4 gift-season build-up. Because many customers are retailers, they stock up *before* Christmas, so the peak arrives a month earlier than consumer retail. Orders and active customers rise together with revenue. The surge is driven by **more customers buying**, not just bigger orders. December 2011 looks like a collapse on the raw chart only because it is a partial month; per trading day it is still strong.

**Business implications:** secure stock and warehouse capacity by August, launch Christmas catalogues and reactivation campaigns in late August/September, and avoid system changes between September and early December.

### 6.3 Time — which weekdays and hours are strongest?

In [ ]:
by_day = (orders.assign(DayOfWeek=pd.Categorical(orders['InvoiceDate'].dt.day_name(), categories=DAY_ORDER, ordered=True))
                .groupby('DayOfWeek', observed=False).agg(Orders=('Revenue', 'size'), Revenue=('Revenue', 'sum')))
by_hour = orders.groupby(orders['InvoiceDate'].dt.hour).agg(Orders=('Revenue', 'size'), Revenue=('Revenue', 'sum'))
by_hour = by_hour.reindex(range(24), fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].bar(by_day.index.astype(str), by_day['Revenue'], color=BLUE, width=0.65)
axes[0].set_title('Revenue by weekday'); gbp_axis(axes[0])
axes[1].bar(by_hour.index, by_hour['Orders'], color=BLUE, width=0.75)
axes[1].set_title('Orders by hour of day'); axes[1].set_xticks(range(0, 24, 2)); axes[1].set_xlabel('Hour')
plt.tight_layout(); plt.show()
display(by_day.T)

In [ ]:
heat = (orders.assign(Day=pd.Categorical(orders['InvoiceDate'].dt.day_name(), categories=DAY_ORDER, ordered=True),
                      Hour=orders['InvoiceDate'].dt.hour)
              .pivot_table(index='Day', columns='Hour', values='Revenue', aggfunc='size', fill_value=0, observed=False))
plt.figure(figsize=(14, 3.8))
sns.heatmap(heat, cmap='Blues', linewidths=0.5, linecolor='white', cbar_kws={'label': 'orders'})
plt.title('Order traffic: weekday × hour'); plt.ylabel(''); plt.show()

quiet = by_hour[by_hour['Orders'] == 0].index.tolist()
print('Hours with zero orders:', quiet)
print('Busiest hour:', by_hour['Orders'].idxmax(), ':00  | Busiest weekday by revenue:', by_day['Revenue'].idxmax())
print('Saturday orders:', int(by_day.loc['Saturday', 'Orders']))

**Interpretation.** Trading follows **UK office hours**: orders start around 07:00–08:00, peak around midday, and stop by about 20:00. **Saturday has no orders at all**, and Sunday is a lighter, shorter day. That pattern fits a B2B customer base that places orders from work.

* **Server maintenance:** schedule overnight, or on **Saturday**, when there is zero traffic.
* **Ad-spend & email timing:** concentrate paid search and email sends from late morning to early afternoon, Tuesday–Thursday. Avoid the weekend for B2B campaigns.

### 6.4 Products — what sells the most units and the most revenue?

In [ ]:
prod = (tx.groupby('StockCode')
          .agg(NetUnits=('Quantity', 'sum'), NetRevenue=('TotalLineRevenue', 'sum'),
               Orders=('InvoiceNo', 'nunique'), Customers=('CustomerID', 'nunique')))
prod['GrossUnits'] = sales.groupby('StockCode')['Quantity'].sum()
prod['Description'] = desc_map.reindex(prod.index)

top_units = prod.nlargest(10, 'NetUnits')
top_rev = prod.nlargest(10, 'NetRevenue')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(top_units['Description'].str[:35][::-1], top_units['NetUnits'][::-1], color=BLUE)
axes[0].set_title('Top 10 products by net units sold')
axes[1].barh(top_rev['Description'].str[:35][::-1], top_rev['NetRevenue'][::-1], color=BLUE)
axes[1].set_title('Top 10 products by net revenue'); gbp_axis(axes[1], 'x')
plt.tight_layout(); plt.show()

print('Products in both top-10 lists:', sorted(set(top_units.index) & set(top_rev.index)))

In [ ]:
# Gross vs net units: which products' volume was inflated by cancelled orders?
prod['CancelledUnitsShare'] = 1 - prod['NetUnits'] / prod['GrossUnits']
prod.nlargest(5, 'GrossUnits')[['Description', 'GrossUnits', 'NetUnits', 'CancelledUnitsShare', 'Orders']]

**Interpretation.** The **volume** leaders are cheap, small items bought by the hundreds (bags, decorations, small craft items). The **revenue** leaders are higher-priced statement items such as the cake stand and the hanging T-light holder. A product can top one list and miss the other, so merchandising should track both.

The gross-versus-net table shows why netting matters. If cancellations were ignored, products whose single huge order was reversed would appear at the top of the volume ranking.

### 6.5 Product concentration — does the 80/20 (Pareto) rule hold?

In [ ]:
p = prod[prod['NetRevenue'] > 0].sort_values('NetRevenue', ascending=False)
cum_rev = p['NetRevenue'].cumsum() / p['NetRevenue'].sum()
cum_prod = np.arange(1, len(p) + 1) / len(p)
share_for_80 = cum_prod[np.searchsorted(cum_rev.values, 0.80)]
rev_top20 = cum_rev.iloc[int(0.2 * len(p)) - 1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cum_prod * 100, cum_rev.values * 100, color=BLUE)
ax.plot([0, 100], [0, 100], color=MUTED, linestyle='--', linewidth=1)
ax.axhline(80, color=MUTED, linewidth=0.8); ax.axvline(share_for_80 * 100, color=MUTED, linewidth=0.8)
ax.annotate(f'{share_for_80:.0%} of products\n→ 80% of revenue', (share_for_80 * 100, 80), xytext=(share_for_80 * 100 + 8, 55),
            arrowprops=dict(arrowstyle='->', color=MUTED))
ax.set_xlabel('% of products (ranked by revenue)'); ax.set_ylabel('% of cumulative net revenue')
ax.set_title('Cumulative revenue curve of StockCodes (Pareto)')
plt.tight_layout(); plt.show()
print(f'Top 20% of products generate {rev_top20:.1%} of net revenue; 80% of revenue comes from {share_for_80:.1%} of products ({int(share_for_80 * len(p)):,} SKUs).')

**Interpretation.** Revenue is highly concentrated: the curve rises steeply, and roughly a fifth of the catalogue carries about 80% of revenue. The printed numbers above give the exact split. These core SKUs must never go out of stock and deserve priority in forecasting. The long tail is a candidate for range rationalisation.

### 6.6 Dead-stock identification

Products with **high historical volume** (top quartile of units sold before the last 90 days) but **zero or negative net sales in the final 90 days**.

In [ ]:
cutoff = df['InvoiceDate'].max() - pd.Timedelta(days=90)
hist_units = tx[tx['InvoiceDate'] < cutoff].groupby('StockCode')['Quantity'].sum()
recent_units = tx[tx['InvoiceDate'] >= cutoff].groupby('StockCode')['Quantity'].sum()

high_volume = hist_units[hist_units >= hist_units.quantile(0.75)]
dead = pd.DataFrame({'HistoricalUnits': high_volume,
                     'Last90DaysNetUnits': recent_units.reindex(high_volume.index).fillna(0)})
dead = dead[dead['Last90DaysNetUnits'] <= 0].sort_values('HistoricalUnits', ascending=False)
dead['Description'] = desc_map.reindex(dead.index)
dead['LastSale'] = sales.groupby('StockCode')['InvoiceDate'].max().reindex(dead.index)
print(f'{len(dead)} high-volume products had no net sales in the final 90 days (since {cutoff:%d %b %Y}).')
dead.head(15)

**Interpretation.** These items sold well historically but stopped selling in the busiest quarter of the year. That suggests they were discontinued, ran out of stock, or are seasonal lines from the previous year. Merchandising should check each one: clear remaining stock with bundles or markdowns, or delist the product so it stops taking warehouse space.

### 6.7 Customers — who contributes the most revenue and orders?

In [ ]:
cust_view = (tx[~tx['Is_Guest']].groupby('CustomerID')
               .agg(NetRevenue=('TotalLineRevenue', 'sum'),
                    Orders=('InvoiceNo', lambda s: s[~s.str.startswith('C')].nunique()),
                    Country=('Country', 'first')))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
t = cust_view.nlargest(10, 'NetRevenue')[::-1]
axes[0].barh('ID ' + t.index + ' (' + t['Country'] + ')', t['NetRevenue'], color=BLUE)
axes[0].set_title('Top 10 customers by net revenue'); gbp_axis(axes[0], 'x')
t = cust_view.nlargest(10, 'Orders')[::-1]
axes[1].barh('ID ' + t.index + ' (' + t['Country'] + ')', t['Orders'], color=BLUE)
axes[1].set_title('Top 10 customers by number of orders')
plt.tight_layout(); plt.show()

cv = cust_view[cust_view['NetRevenue'] > 0].sort_values('NetRevenue', ascending=False)
c_cum = cv['NetRevenue'].cumsum() / cv['NetRevenue'].sum()
print(f"Top 1% of customers  -> {c_cum.iloc[max(int(0.01 * len(cv)) - 1, 0)]:.1%} of registered net revenue")
print(f"Top 10% of customers -> {c_cum.iloc[int(0.10 * len(cv)) - 1]:.1%}")
print(f"Top 20% of customers -> {c_cum.iloc[int(0.20 * len(cv)) - 1]:.1%}")
print(f"One-time buyers: {(cust_view['Orders'] == 1).mean():.1%} of customers")

**Interpretation.** Customer value is even more concentrated than product value. A few dozen accounts, several of them international wholesalers, generate a disproportionate share of revenue. The top revenue customers and the top order-count customers are **not the same people**: some accounts buy rarely but in huge volumes, while others order very often in smaller amounts. This is why we need *both* Frequency and Monetary in RFM. A large share of customers bought only once, so converting first-time buyers into repeat buyers is a major lever.

### 6.8 Geography — which countries generate customers, orders and revenue?

In [ ]:
geo = (orders.groupby('Country')
             .agg(Orders=('Revenue', 'size'), GrossRevenue=('Revenue', 'sum'), AOV=('Revenue', 'mean'),
                  UnitsPerOrder=('Units', 'mean')))
geo['Customers'] = sales.groupby('Country')['CustomerID'].nunique()
geo['NetRevenue'] = tx.groupby('Country')['TotalLineRevenue'].sum()
geo['RevenueShare_%'] = geo['NetRevenue'] / geo['NetRevenue'].sum() * 100
geo = geo.sort_values('NetRevenue', ascending=False)
display(geo.head(12))

uk_vs = (orders.groupby(orders['Is_UK'].map({True: 'United Kingdom', False: 'International'}))
               .agg(Orders=('Revenue', 'size'), Revenue=('Revenue', 'sum'), AOV=('Revenue', 'mean'),
                    MedianOrder=('Revenue', 'median'), UnitsPerOrder=('Units', 'mean')))
uk_vs['Customers'] = sales.groupby(sales['Is_UK'].map({True: 'United Kingdom', False: 'International'}))['CustomerID'].nunique()
for col in ['Orders', 'Revenue', 'Customers']:
    uk_vs[f'{col}_share_%'] = uk_vs[col] / uk_vs[col].sum() * 100
display(uk_vs)

In [ ]:
intl = geo.drop(index='United Kingdom', errors='ignore').head(10)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
for ax, col, title in zip(axes, ['Customers', 'Orders', 'NetRevenue'],
                          ['Customers', 'Orders', 'Net revenue']):
    ax.barh(intl.index[::-1], intl[col][::-1], color=BLUE)
    ax.set_title(f'{title} — top 10 outside the UK')
gbp_axis(axes[2], 'x')
plt.tight_layout(); plt.show()

**Interpretation.** The UK dominates customers, orders and revenue, which is why we use the `Is_UK` flag. Outside the UK the picture is different: a few markets (the Netherlands, EIRE, Germany, France, Australia) generate most international revenue, often from **very few customers with very large orders**. International orders have a clearly higher average value than UK orders. Abroad, the business mainly serves **distributors and wholesalers**, while at home it serves many small shops.

* Treat key international accounts (e.g. the Netherlands and Australia) as *key-account* relationships with dedicated service.
* Germany and France have many customers with moderate spend. They are the best candidates for localised marketing to grow the customer count.

### 6.9 Baskets — is there wholesale / bulk-buying behaviour?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
ov = orders.loc[orders['Revenue'] > 0, 'Revenue']
axes[0].hist(np.log10(ov), bins=60, color=BLUE)
axes[0].set_title('Order value distribution (log scale)'); axes[0].set_xlabel('order value (£)')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'£{10**v:,.0f}'))
axes[1].hist(np.log10(orders.loc[orders['Units'] > 0, 'Units']), bins=60, color=BLUE)
axes[1].set_title('Units per order (log scale)'); axes[1].set_xlabel('units')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{10**v:,.0f}'))
plt.tight_layout(); plt.show()

display(orders[['Revenue', 'Units', 'Lines', 'DistinctProducts']].describe(percentiles=[.25, .5, .75, .9, .95, .99]).T)

In [ ]:
# Evidence 1: line quantities cluster on trade-pack sizes (multiples of 12)
q = sales['Quantity'].value_counts().head(15)
print('Most common line quantities:', dict(q))
print(f"Share of lines with quantity that is a multiple of 12: {(sales['Quantity'] % 12 == 0).mean():.1%}")

# Evidence 2: define a 'bulk order' as the top 5% of orders by units
bulk_threshold = orders['Units'].quantile(0.95)
orders['IsBulk'] = orders['Units'] >= bulk_threshold
print(f"\nBulk-order threshold (95th percentile): {bulk_threshold:,.0f} units")
print(f"Bulk orders = {orders['IsBulk'].mean():.1%} of orders but {orders.loc[orders['IsBulk'], 'Revenue'].sum() / orders['Revenue'].sum():.1%} of revenue")
print(f"Share of bulk orders placed from outside the UK: {(~orders.loc[orders['IsBulk'], 'Is_UK']).mean():.1%} "
      f"(vs {(~orders['Is_UK']).mean():.1%} of all orders)")

**Interpretation.** Both order value and units per order span several orders of magnitude. The most common line quantities are **trade-pack sizes** (multiples of 6, 12, 24…), and the top 5% of orders by volume bring in a far larger share of revenue. This is clear evidence of a **mixed B2B / B2C customer base**: a long tail of small orders plus a core of trade buyers. Two consequences:

1. **Average order value is misleading**: report the median alongside it.
2. Raw Monetary and Frequency values are **heavily right-skewed**, so clustering will need a log transform.

### 6.10 Returns — how common are cancellations and who drives them?

In [ ]:
n_cancel_inv = returns['InvoiceNo'].nunique()
print(f"Cancellation invoices: {n_cancel_inv:,} = {n_cancel_inv / (n_cancel_inv + len(orders)):.1%} of all invoices")
print(f"Cancellation lines: {len(returns):,} = {len(returns) / len(tx):.1%} of product lines")
print(f"Value refunded: {gbp(refunds)} = {refunds / gross:.1%} of gross revenue")
print(f"Customers with at least one cancellation: {returns['CustomerID'].nunique():,} "
      f"of {sales['CustomerID'].nunique():,} ({returns['CustomerID'].nunique() / sales['CustomerID'].nunique():.1%})")

ret_prod = pd.DataFrame({'GrossUnits': sales.groupby('StockCode')['Quantity'].sum(),
                         'ReturnedUnits': -returns.groupby('StockCode')['Quantity'].sum(),
                         'ReturnValue': -returns.groupby('StockCode')['TotalLineRevenue'].sum()}).fillna(0)
ret_prod['ReturnRate_%'] = ret_prod['ReturnedUnits'] / ret_prod['GrossUnits'].replace(0, np.nan) * 100
ret_prod['Description'] = desc_map.reindex(ret_prod.index)
print('\nTop products by value returned:')
display(ret_prod.nlargest(8, 'ReturnValue'))
print('Highest return rates among products with >= 500 units sold:')
display(ret_prod[ret_prod['GrossUnits'] >= 500].nlargest(8, 'ReturnRate_%'))

In [ ]:
ret_cust = pd.DataFrame({'GrossRevenue': sales[~sales['Is_Guest']].groupby('CustomerID')['TotalLineRevenue'].sum(),
                         'ReturnValue': -returns.groupby('CustomerID')['TotalLineRevenue'].sum()}).fillna(0)
ret_cust['ReturnRate_%'] = ret_cust['ReturnValue'] / ret_cust['GrossRevenue'].replace(0, np.nan) * 100
print('Customers with the highest value returned:')
display(ret_cust.nlargest(8, 'ReturnValue'))

ret_geo = pd.DataFrame({'Gross': sales.groupby('Country')['TotalLineRevenue'].sum(),
                        'Returned': -returns.groupby('Country')['TotalLineRevenue'].sum()}).fillna(0)
ret_geo = ret_geo[ret_geo['Gross'] > 50_000]
ret_geo['ReturnRate_%'] = ret_geo['Returned'] / ret_geo['Gross'] * 100

monthly_ret = -returns.groupby('YearMonth')['TotalLineRevenue'].sum()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].bar(monthly_ret.index.astype(str), monthly_ret.values, color=ORANGE)
axes[0].set_title('Value refunded per month'); gbp_axis(axes[0]); axes[0].tick_params(axis='x', rotation=60)
r = ret_geo['ReturnRate_%'].sort_values()
axes[1].barh(r.index, r.values, color=ORANGE)
axes[1].set_title('Return rate by country (countries with > £50k gross)'); axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

**Interpretation.** Cancellations are frequent at invoice level but small at value level for most customers. The refunded value is driven by **a few extreme reversals**, typically large orders cancelled within minutes, plus a group of repeat-cancelling accounts. Return monitoring should therefore:

* flag *orders* above a size threshold for manual confirmation before fulfilment (would have caught the giant reversed orders);
* watch customers whose return rate is persistently high; this is a cost-to-serve issue, not a loyalty signal;
* review products with high return *rates* (not just high return values) for quality or description problems.

### 6.11 What does a typical customer look like?

In [ ]:
c_orders = orders[~orders['Is_Guest']].groupby('CustomerID')
typical = pd.DataFrame({
    'Orders': c_orders.size(),
    'NetRevenue': cust_view['NetRevenue'],
    'AOV': c_orders['Revenue'].mean(),
    'DistinctProducts': sales[~sales['Is_Guest']].groupby('CustomerID')['StockCode'].nunique(),
    'TenureDays': (c_orders['InvoiceDate'].max() - c_orders['InvoiceDate'].min()).dt.days,
    'DaysSinceLastOrder': (df['InvoiceDate'].max() - c_orders['InvoiceDate'].max()).dt.days,
}).dropna()
summary = typical.describe(percentiles=[.25, .5, .75, .9]).T[['25%', '50%', 'mean', '75%', '90%']]
summary.columns = ['P25', 'Median (typical)', 'Mean', 'P75', 'P90']
summary

**Interpretation.** The *median* customer, the honest "typical" customer, places a handful of orders a year, spends a moderate amount per order, buys dozens of different products, and last ordered weeks to a few months before the data ends. The mean is pulled far above the median by trade accounts on every measure. So any segmentation must handle extreme values, and segment descriptions should quote **medians**.

## 7. Business Insights

Each insight below is computed from the data (run the cell), followed by what the business should do with it. These are **descriptive** findings. They show *what* happened, not *why*, and they are not causal claims.

In [ ]:
best_day = by_day['Revenue'].idxmax()
peak_hour = by_hour['Orders'].idxmax()
top_country_intl = geo.drop(index='United Kingdom', errors='ignore').index[0]
uk_rev_share = uk_vs.loc['United Kingdom', 'Revenue_share_%'] if 'United Kingdom' in uk_vs.index else np.nan
aov_ratio = uk_vs.loc['International', 'AOV'] / uk_vs.loc['United Kingdom', 'AOV']

insights = [
    f"Seasonality: revenue peaks in {monthly['NetRevenue'].idxmax()}; Sep–Nov delivers {q4_share:.0%} of Jan–Nov 2011 revenue.",
    f"Growth is customer-driven: active customers rise from {int(monthly['ActiveCustomers'].iloc[0]):,} to a peak of {int(monthly['ActiveCustomers'].max()):,} per month.",
    f"Weekly rhythm: {best_day} is the strongest revenue day; Saturday has {int(by_day.loc['Saturday', 'Orders'])} orders.",
    f"Daily rhythm: orders peak at {peak_hour}:00; hours with zero orders: {quiet[:3]}...{quiet[-3:]} -> safe maintenance windows.",
    f"Product concentration: {share_for_80:.0%} of products generate 80% of net revenue.",
    f"Customer concentration: top 10% of customers generate {c_cum.iloc[int(0.10 * len(cv)) - 1]:.0%} of registered revenue.",
    f"Retention gap: {(cust_view['Orders'] == 1).mean():.0%} of customers ordered only once.",
    f"Domestic dominance: the UK generates {uk_rev_share:.0f}% of revenue; top international market is {top_country_intl}.",
    f"International orders are {aov_ratio:.1f}x the value of UK orders on average (wholesale/distributor profile).",
    f"Bulk buying: top 5% of orders by units = {orders.loc[orders['IsBulk'], 'Revenue'].sum() / orders['Revenue'].sum():.0%} of revenue; "
    f"{(sales['Quantity'] % 12 == 0).mean():.0%} of lines are multiples of 12 (trade packs).",
    f"Returns: {refunds / gross:.1%} of gross revenue refunded; {n_cancel_inv / (n_cancel_inv + len(orders)):.0%} of invoices are cancellations.",
    f"Dead stock: {len(dead)} formerly high-volume SKUs had no net sales in the final 90 days.",
    f"Guest checkout: {orders.loc[orders['Is_Guest'], 'Revenue'].sum() / gross:.0%} of revenue comes from orders without a CustomerID — invisible to CRM.",
]
for i, s in enumerate(insights, 1):
    print(f'{i:>2}. {s}')

| # | Insight | So what — recommended action |
|---|---|---|
| 1 | Strong Sep–Nov peak (Q4 gifting, retailers stock up early) | Lock in inventory & staffing by August; launch Christmas catalogue in late August |
| 2 | The peak is driven by *more active customers*, not only bigger orders | Run reactivation campaigns in August–September to maximise the number of buyers in season |
| 3 | Mid-week is strongest; no Saturday trading | Schedule campaigns Tue–Thu; use Saturday for deployments and maintenance |
| 4 | Orders follow office hours, peaking around midday | Time emails and paid search for late morning; run batch jobs overnight |
| 5 | ~20% of SKUs drive ~80% of revenue | Protect core SKUs with safety stock; review the long tail for delisting |
| 6 | A small group of customers drives most revenue | Key-account management for the top accounts; monitor them for churn signals |
| 7 | A large share of customers buys only once | Second-purchase onboarding journey is the biggest retention lever |
| 8 | UK dominates; a few international markets are significant | Analyse UK and international separately (`Is_UK`) so the UK doesn't hide international trends |
| 9 | International orders are much larger | International = distributors: offer volume pricing, trade terms, dedicated account managers |
| 10 | Clear trade-pack and bulk-order behaviour | Formalise a wholesale programme (tiered pricing, pack sizes, trade portal) |
| 11 | Cancellations are frequent; value driven by a few huge reversals | Confirm very large orders before fulfilment; track high-return customers and products |
| 12 | Formerly popular SKUs stopped selling | Clear or delist dead stock to free warehouse capacity before peak |
| 13 | Guest checkout revenue is untracked | Encourage account creation at checkout so these buyers can be included in CRM |

## 8. Customer-Level Feature Engineering

From here on the unit of analysis is the **customer**. We move from transaction lines to one row per registered customer.

* **Snapshot date** = last transaction date + 1 day (the "today" from which recency is measured).
* **Frequency** counts unique *purchase* invoices (orders), not lines and not cancellations.
* **Monetary** is **net** spend (purchases minus refunds).
* Supporting features (AOV, tenure, products, return rate, country) are used later to *describe* segments, not to build them.

In [ ]:
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)
cs = sales[~sales['Is_Guest']]
cr = returns[~returns['Is_Guest']]

customers = cs.groupby('CustomerID').agg(
    FirstPurchase=('InvoiceDate', 'min'), LastPurchase=('InvoiceDate', 'max'),
    Frequency=('InvoiceNo', 'nunique'), GrossRevenue=('TotalLineRevenue', 'sum'),
    Units=('Quantity', 'sum'), DistinctProducts=('StockCode', 'nunique'),
    Country=('Country', lambda s: s.mode().iat[0]))
customers['ReturnValue'] = (-cr.groupby('CustomerID')['TotalLineRevenue'].sum()).reindex(customers.index).fillna(0)
customers['Monetary'] = customers['GrossRevenue'] - customers['ReturnValue']
customers['Recency'] = (snapshot - customers['LastPurchase']).dt.days
customers['TenureDays'] = (snapshot - customers['FirstPurchase']).dt.days
customers['AOV'] = customers['GrossRevenue'] / customers['Frequency']
customers['ReturnRate'] = customers['ReturnValue'] / customers['GrossRevenue']
customers['Is_UK'] = customers['Country'].eq('United Kingdom')

n_nonpos = (customers['Monetary'] <= 0).sum()
print(f'Snapshot date: {snapshot:%d %b %Y}')
print(f'Customers with at least one purchase: {len(customers):,}')
print(f'Customers with net Monetary <= 0 (refunds >= purchases in the window): {n_nonpos} -> excluded from RFM/clustering')

rfm = customers[customers['Monetary'] > 0].copy()
rfm[['Recency', 'Frequency', 'Monetary', 'AOV', 'TenureDays', 'DistinctProducts', 'ReturnRate', 'Country']].head()

## 9. RFM Analysis

| Dimension | Definition | Direction |
|---|---|---|
| **Recency (R)** | Days since the customer's last purchase | **Lower is better** → score reversed (5 = most recent) |
| **Frequency (F)** | Number of unique purchase invoices | Higher is better |
| **Monetary (M)** | Net revenue generated | Higher is better |

**Why RFM matters to the business:**
* **Actionable personalisation**: stops over-discounting loyal buyers while targeting retention offers at customers who are leaving.
* **Churn early-warning**: a falling R score shows disengagement before the account is lost.
* **Lifetime value protection**: separates high-value trade accounts from small one-off buyers.

**Scoring.** Each metric is split into 5 equal-sized groups (quintiles) with `pd.qcut`. Many customers have the same Frequency (lots of 1s and 2s), so equal-width bins are impossible. We rank first (`rank(method='first')`) so each quintile gets an equal number of customers. The trade-off is that two customers with the same frequency can sit in neighbouring F scores; the boundary table below makes that visible.

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)  # e.g. '555'
rfm['RFM_Sum'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# What does each score mean in real units?
bounds = pd.concat({
    'Recency (days)': rfm.groupby('R_Score')['Recency'].agg(['min', 'max']),
    'Frequency (orders)': rfm.groupby('F_Score')['Frequency'].agg(['min', 'max']),
    'Monetary (£)': rfm.groupby('M_Score')['Monetary'].agg(['min', 'max']),
}, axis=1)
bounds.index.name = 'score'
display(bounds)
print('Top-tier "555" customers:', (rfm['RFM_Score'] == '555').sum())

**Reading the scores.** An R score of 5 means the customer bought in the most recent fifth of all customers (only days before the snapshot). An R score of 1 means the customer has not bought for many months. F and M work the other way round: 5 is the top fifth by orders and by net spend. `555` is a top-tier customer on all three dimensions.

### Rule-based RFM segments

Segments follow the business translation matrix in the guideline. The core four come straight from it; three supporting groups make sure every customer lands somewhere meaningful.

In [ ]:
def rfm_segment(r):
    R, F, M = r['R_Score'], r['F_Score'], r['M_Score']
    if R >= 4 and F >= 4 and M >= 4:  return 'Champions'               # recent, frequent, high spend
    if R <= 2 and F >= 3 and M >= 3:  return 'At-Risk High-Value'      # used to buy a lot, stopped
    if R >= 3 and F >= 3 and M >= 3:  return 'Loyal Customers'         # consistent regular spenders
    if R >= 4 and F <= 2:             return 'New Inflow'              # recent first-time / low-frequency buyers
    if R == 3 and F <= 2:             return 'Promising'               # fairly recent, not yet repeat
    if R <= 2 and F <= 2 and M <= 2:  return 'Hibernating / Lost'      # long gone, low value
    return 'Needs Attention'                                           # mixed signals

rfm['RFM_Segment'] = rfm.apply(rfm_segment, axis=1)

seg_order = ['Champions', 'Loyal Customers', 'Promising', 'New Inflow', 'Needs Attention', 'At-Risk High-Value', 'Hibernating / Lost']
seg_tbl = (rfm.groupby('RFM_Segment')
              .agg(Customers=('Monetary', 'size'), Median_Recency=('Recency', 'median'),
                   Median_Frequency=('Frequency', 'median'), Median_Monetary=('Monetary', 'median'),
                   Revenue=('Monetary', 'sum'))
              .reindex(seg_order).dropna(how='all'))
seg_tbl['Customers_%'] = seg_tbl['Customers'] / seg_tbl['Customers'].sum() * 100
seg_tbl['Revenue_%'] = seg_tbl['Revenue'] / seg_tbl['Revenue'].sum() * 100
seg_tbl

## 10. RFM Visualization & Interpretation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    ax.hist(rfm[col].clip(upper=rfm[col].quantile(0.99)), bins=50, color=BLUE)
    ax.set_title(f'{col}  (skew = {rfm[col].skew():.1f})')
axes[2].xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'£{v/1e3:.0f}k'))
fig.suptitle('Raw RFM distributions (clipped at 99th percentile for display)', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
s = seg_tbl.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
axes[0].barh(s.index, s['Customers_%'], color=BLUE); axes[0].set_title('Share of customers')
axes[1].barh(s.index, s['Revenue_%'], color=BLUE); axes[1].set_title('Share of net revenue')
for ax, col in zip(axes, ['Customers_%', 'Revenue_%']):
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    for yv, v in zip(s.index, s[col]):
        ax.text(v + 0.5, yv, f'{v:.0f}%', va='center', color='#52514e', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
pivot = rfm.pivot_table(index='R_Score', columns='F_Score', values='Monetary', aggfunc='median').sort_index(ascending=False)
counts = rfm.pivot_table(index='R_Score', columns='F_Score', values='Monetary', aggfunc='size').sort_index(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='Blues', ax=axes[0], cbar_kws={'label': 'median £'})
axes[0].set_title('Median Monetary (£) by R and F score')
sns.heatmap(counts, annot=True, fmt=',.0f', cmap='Blues', ax=axes[1], cbar_kws={'label': 'customers'})
axes[1].set_title('Number of customers by R and F score')
plt.tight_layout(); plt.show()

**Interpretation.**

* **The raw R, F, M distributions are heavily right-skewed**, especially Frequency and Monetary. A small number of trade accounts have extreme values, which confirms the wholesale finding from the EDA.
* **Frequency and Monetary move together.** Median spend rises steadily from left to right in the heatmap, so customers who order more often are also worth more. Recency adds separate information: the bottom-right cells (old *and* frequent) are high-value customers we are losing.
* **Champions and Loyal Customers** are a minority of customers but generate the majority of revenue. **At-Risk High-Value** customers hold a meaningful share of past revenue; they are the best target for win-back spend. **Hibernating / Lost** is a large group by headcount and a small one by revenue, so cheap, automated contact is appropriate there.

Rule-based RFM is transparent but relies on cut-offs that *we* chose. Next we check whether the data forms these groups *on its own* by clustering.

## 11. Preprocessing for Clustering

1. **Features:** raw `Recency`, `Frequency`, `Monetary`. **`CustomerID` is never a feature**, because it is an arbitrary label. We cluster on the raw values rather than the 1–5 scores, because scores throw away information and bake in our own cut-offs.
2. **Skewness transformation:** `log1p` compresses the heavy right tail from wholesale volumes, so a few whales don't define the clusters.
3. **Standardisation:** `StandardScaler` → mean 0, variance 1. Without it, Monetary (in thousands of £) would dwarf Recency (in days) and Frequency (single digits) in the Euclidean distance.

In [ ]:
FEATURES = ['Recency', 'Frequency', 'Monetary']
X_raw = rfm[FEATURES].copy()
X_log = np.log1p(X_raw)

skew_tbl = pd.DataFrame({'skew_before': X_raw.skew(), 'skew_after_log': X_log.skew()})
display(skew_tbl)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_log), index=rfm.index, columns=FEATURES)
display(X_scaled.agg(['mean', 'std']).round(3))   # check: mean ~ 0, std ~ 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=(X_raw - X_raw.mean()) / X_raw.std(), ax=axes[0], color=BLUE, fliersize=2)
axes[0].set_title('Raw (z-scored only): extreme outliers dominate')
sns.boxplot(data=X_scaled, ax=axes[1], color=BLUE, fliersize=2)
axes[1].set_title('log1p + StandardScaler: comparable scales')
plt.tight_layout(); plt.show()

After the log transform, skewness falls from very high values to near zero for Frequency and Monetary. After scaling, all three features have mean 0 and standard deviation 1, so each contributes equally to distance calculations.

## 12. Clustering & Evaluation

We compare three algorithms and use several metrics, so we don't choose *K* only because an elbow "looks right":

| Algorithm | Strengths / weaknesses | Role here |
|---|---|---|
| **K-Means** | Fast, interpretable / sensitive to outliers, assumes round clusters | Baseline segmentation |
| **Hierarchical (Ward)** | No fixed K needed, dendrogram / expensive on large data | Check the cluster structure and the choice of K |
| **DBSCAN** | Isolates extreme outliers (whale buyers), any cluster shape / struggles with varying density | Detect anomalous accounts |

### 12.1 K-Means — choosing K

In [ ]:
K_RANGE = range(2, 11)
rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit(X_scaled)
    rows.append({'k': k, 'WCSS': km.inertia_,
                 'silhouette': silhouette_score(X_scaled, km.labels_),
                 'davies_bouldin': davies_bouldin_score(X_scaled, km.labels_),
                 'calinski_harabasz': calinski_harabasz_score(X_scaled, km.labels_)})
k_eval = pd.DataFrame(rows).set_index('k')
display(k_eval)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(k_eval.index, k_eval['WCSS'], marker='o', color=BLUE); axes[0].set_title('Elbow: WCSS (lower = tighter)')
axes[1].plot(k_eval.index, k_eval['silhouette'], marker='o', color=BLUE); axes[1].set_title('Silhouette (higher = better separated)')
axes[2].plot(k_eval.index, k_eval['davies_bouldin'], marker='o', color=BLUE); axes[2].set_title('Davies–Bouldin (lower = better)')
for ax in axes: ax.set_xlabel('k'); ax.set_xticks(list(K_RANGE))
plt.tight_layout(); plt.show()

In [ ]:
# Elbow point = largest second difference of WCSS (where the drop-off starts to plateau)
wcss = k_eval['WCSS']
elbow_k = int((wcss.diff() - wcss.diff().shift(-1)).idxmax())

# Business constraint: 3–6 segments is what marketing can realistically run campaigns for.
candidates = k_eval.loc[3:6]
best_sil_k = int(candidates['silhouette'].idxmax())
K = best_sil_k
print(f'Elbow (max curvature) at k = {elbow_k}')
print(f'Best silhouette within actionable range 3–6: k = {best_sil_k} (silhouette = {candidates.loc[best_sil_k, "silhouette"]:.3f})')
print(f'Global best silhouette: k = {int(k_eval["silhouette"].idxmax())} (k=2 usually wins mechanically but only splits "good" vs "bad")')
print(f'==> Chosen K = {K}')

# Stability: does the solution change with the random initialisation?
base = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE).fit_predict(X_scaled)
aris = [adjusted_rand_score(base, KMeans(n_clusters=K, n_init=1, random_state=s).fit_predict(X_scaled)) for s in range(10)]
print(f'Stability across 10 random seeds: mean ARI = {np.mean(aris):.3f} (1.0 = identical clustering)')

**How K was chosen (three pieces of evidence, not one plot):**
1. **Elbow:** WCSS drops sharply for small K, then the gains flatten. Adding clusters beyond the elbow buys little extra compactness.
2. **Silhouette & Davies–Bouldin:** K = 2 usually scores best mechanically, but it only splits customers into "good" and "bad", which is not actionable. Within the **3–6 range** that marketing can actually run campaigns for, we take the K with the best separation.
3. **Stability:** re-running K-Means with different random starts gives (almost) the same clusters (ARI close to 1). The segments are a real structure, not an artefact of initialisation.

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=50, random_state=RANDOM_STATE).fit(X_scaled)
rfm['KMeans'] = kmeans.labels_

# 2-D view of the clusters (PCA is only for plotting — clustering used all 3 scaled features)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pc = pca.fit_transform(X_scaled)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for c in sorted(rfm['KMeans'].unique()):
    m = rfm['KMeans'].values == c
    axes[0].scatter(pc[m, 0], pc[m, 1], s=8, alpha=0.6, color=PALETTE[c], label=f'Cluster {c}')
    axes[1].scatter(rfm.loc[m, 'Recency'], rfm.loc[m, 'Monetary'], s=8, alpha=0.6, color=PALETTE[c], label=f'Cluster {c}')
axes[0].set_title(f'K-Means clusters in PCA space ({pca.explained_variance_ratio_.sum():.0%} of variance shown)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend(markerscale=3)
axes[1].set_yscale('log'); axes[1].set_xlabel('Recency (days)'); axes[1].set_ylabel('Monetary (£, log scale)')
axes[1].set_title('Clusters on original units'); axes[1].legend(markerscale=3)
plt.tight_layout(); plt.show()

### 12.2 Hierarchical clustering (Ward)

A full dendrogram for thousands of customers is unreadable and expensive, so we draw it on a random sub-sample to see where the tree is best cut. Then we fit Ward clustering on all customers with the same K for comparison.

In [ ]:
sample = X_scaled.sample(n=min(800, len(X_scaled)), random_state=RANDOM_STATE)
Z = linkage(sample, method='ward')
plt.figure(figsize=(14, 4.5))
dendrogram(Z, truncate_mode='lastp', p=30, color_threshold=Z[-(K - 1), 2], above_threshold_color=MUTED, no_labels=True)
plt.axhline(Z[-(K - 1), 2], color=ORANGE, linestyle='--', linewidth=1)
plt.title(f'Ward dendrogram (sample of {len(sample)} customers); dashed line = cut giving K = {K}')
plt.ylabel('merge distance'); plt.grid(False); plt.show()

gaps = np.diff(Z[::-1, 2])[:8] * -1
print('Largest jumps in merge distance at k =', [int(i) + 2 for i in np.argsort(gaps)[::-1][:3]], '(a big jump = natural cut)')

hier = AgglomerativeClustering(n_clusters=K, linkage='ward').fit(X_scaled)
rfm['Hierarchical'] = hier.labels_
print(f'Ward silhouette (K={K}): {silhouette_score(X_scaled, hier.labels_):.3f}')
print(f'Agreement with K-Means (ARI): {adjusted_rand_score(rfm["KMeans"], rfm["Hierarchical"]):.3f}')

### 12.3 DBSCAN — finding the whales

DBSCAN needs `eps` (neighbourhood radius) and `min_samples`. We use `min_samples = 2 × number of features = 6` and read `eps` from the k-distance curve, at the point where distances start rising steeply.

In [ ]:
MIN_SAMPLES = 2 * X_scaled.shape[1]
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_scaled)
kdist = np.sort(nn.kneighbors(X_scaled)[0][:, -1])
EPS = float(np.percentile(kdist, 97))       # knee region of the k-distance curve

plt.figure(figsize=(9, 4))
plt.plot(kdist, color=BLUE); plt.axhline(EPS, color=ORANGE, linestyle='--', linewidth=1)
plt.title(f'k-distance plot (k = {MIN_SAMPLES}); dashed line = chosen eps {EPS:.2f}')
plt.xlabel('customers sorted by distance'); plt.ylabel(f'distance to {MIN_SAMPLES}th neighbour'); plt.show()

db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(X_scaled)
rfm['DBSCAN'] = db.labels_
n_noise = (db.labels_ == -1).sum()
print(f'DBSCAN clusters: {len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)} | noise/outlier customers: {n_noise} ({n_noise / len(rfm):.1%})')

outl = rfm[rfm['DBSCAN'] == -1]
comp = pd.DataFrame({'Outliers (DBSCAN noise)': outl[FEATURES].median(), 'Everyone else': rfm[rfm['DBSCAN'] != -1][FEATURES].median()})
display(comp)
print(f"Outliers hold {outl['Monetary'].sum() / rfm['Monetary'].sum():.1%} of revenue")
display(outl.nlargest(10, 'Monetary')[['Country', 'Recency', 'Frequency', 'Monetary', 'AOV']])

**Interpretation.** DBSCAN puts almost everyone in one dense cloud. Customer value is a *continuum*, not separate islands. The points it rejects as noise are the extremes: mainly **whale accounts** with very high frequency and spend, plus a few unusual one-off buyers. This is where DBSCAN helps. It does not produce marketing segments, but it **flags the accounts that need individual key-account treatment** rather than a segment campaign.

### 12.4 Comparison

In [ ]:
def sil_or_nan(labels, X):
    mask = labels != -1
    return silhouette_score(X[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan

comparison = pd.DataFrame({
    'K-Means': {'clusters': K, 'silhouette': silhouette_score(X_scaled, rfm['KMeans']),
                'davies_bouldin': davies_bouldin_score(X_scaled, rfm['KMeans']), 'outliers_flagged': 0},
    'Hierarchical (Ward)': {'clusters': K, 'silhouette': silhouette_score(X_scaled, rfm['Hierarchical']),
                            'davies_bouldin': davies_bouldin_score(X_scaled, rfm['Hierarchical']), 'outliers_flagged': 0},
    'DBSCAN': {'clusters': len(set(db.labels_)) - (1 if -1 in db.labels_ else 0),
               'silhouette': sil_or_nan(rfm['DBSCAN'].values, X_scaled.values),
               'davies_bouldin': np.nan, 'outliers_flagged': n_noise},
}).T
comparison

**Decision.** We use **K-Means** for the final segmentation: it separates customers well, is stable, agrees closely with hierarchical clustering (so the structure is not algorithm-specific), and is easy to explain and to re-run every month. **DBSCAN's outlier flag** is kept as an extra column, so whale accounts can be routed to key-account management.

## 13. Segment Profiling

Cluster numbers mean nothing to a marketing team. Below we describe each cluster in real units (medians), compare it with the overall customer base, and **name it from its statistics**. A cluster is labelled "recent", "frequent" or "high-spend" only when its average on that (standardised) feature is better than the customer-base average.

In [ ]:
prof = rfm.groupby('KMeans').agg(
    Customers=('Monetary', 'size'),
    Recency_med=('Recency', 'median'), Frequency_med=('Frequency', 'median'), Monetary_med=('Monetary', 'median'),
    AOV_med=('AOV', 'median'), Tenure_med=('TenureDays', 'median'), Products_med=('DistinctProducts', 'median'),
    ReturnRate_mean=('ReturnRate', 'mean'), UK_share=('Is_UK', 'mean'), Avg_RFM_Sum=('RFM_Sum', 'mean'),
    Revenue=('Monetary', 'sum'))
prof['Customers_%'] = prof['Customers'] / prof['Customers'].sum() * 100
prof['Revenue_%'] = prof['Revenue'] / prof['Revenue'].sum() * 100

overall = rfm[FEATURES].median()
# Naming uses each cluster's mean on the standardised (log) features: > 0 means above the customer-base average.
# (Medians tie too often on Frequency, e.g. many clusters have a median of 2 orders.)
z = X_scaled.groupby(rfm['KMeans']).mean()
prof[['zR', 'zF', 'zM']] = z[FEATURES].values

def base_name(r):
    recent = r['zR'] < 0          # lower Recency = more recent
    frequent = r['zF'] > 0
    big = r['zM'] > 0
    if recent and frequent and big:       return 'Champions / Loyal'
    if not recent and (frequent or big):  return 'At-Risk High-Value'
    if recent and not frequent:           return 'New Inflow'
    if not recent and not frequent and not big: return 'Hibernating / Lost'
    return 'Needs Attention'

prof['base'] = prof.apply(base_name, axis=1)
names = {}
for b, grp in prof.sort_values('Monetary_med', ascending=False).groupby('base', sort=False):
    ids = grp.index.tolist()
    if b == 'Champions / Loyal':
        labels = ['Champions'] + ['Loyal Customers' + ('' if i == 0 else f' {i + 1}') for i in range(len(ids) - 1)]
    else:
        labels = [b] if len(ids) == 1 else [f'{b} (tier {i + 1})' for i in range(len(ids))]
    names.update(dict(zip(ids, labels)))
prof['Segment'] = prof.index.map(names)
rfm['Segment'] = rfm['KMeans'].map(names)

prof = prof.sort_values('Avg_RFM_Sum', ascending=False)
display(prof.drop(columns=['base', 'zR', 'zF', 'zM']).set_index('Segment', append=True)
            .style.format({'Monetary_med': '£{:,.0f}', 'AOV_med': '£{:,.0f}', 'Revenue': '£{:,.0f}',
                           'ReturnRate_mean': '{:.1%}', 'UK_share': '{:.0%}', 'Customers_%': '{:.1f}%', 'Revenue_%': '{:.1f}%',
                           'Recency_med': '{:.0f}', 'Frequency_med': '{:.0f}', 'Tenure_med': '{:.0f}', 'Products_med': '{:.0f}',
                           'Avg_RFM_Sum': '{:.1f}', 'Customers': '{:,}'}))

In [ ]:
seg_list = prof['Segment'].tolist()
colors = {s: PALETTE[i] for i, s in enumerate(seg_list)}

# Snake plot: average standardised (log) value of each feature per segment
snake = X_scaled.assign(Segment=rfm['Segment']).groupby('Segment')[FEATURES].mean().reindex(seg_list)
# Relative importance: segment median vs overall median (−1 … +∞)
rel = (rfm.groupby('Segment')[FEATURES].median() / overall - 1).reindex(seg_list)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for sname in seg_list:
    axes[0].plot(FEATURES, snake.loc[sname], marker='o', color=colors[sname], label=sname)
axes[0].axhline(0, color=MUTED, linewidth=0.8)
axes[0].set_title('Snake plot (standardised means; Recency: higher = OLDER)'); axes[0].legend(fontsize=9)
sns.heatmap(rel, annot=True, fmt='+.0%', cmap='RdBu_r', center=0, ax=axes[1], cbar=False)
axes[1].set_title('Segment median vs overall median'); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
p2 = prof.set_index('Segment').iloc[::-1]
axes[0].barh(p2.index, p2['Customers_%'], color=[colors[s] for s in p2.index]); axes[0].set_title('Share of customers')
axes[1].barh(p2.index, p2['Revenue_%'], color=[colors[s] for s in p2.index]); axes[1].set_title('Share of net revenue')
for ax, col in zip(axes, ['Customers_%', 'Revenue_%']):
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    for yv, v in zip(p2.index, p2[col]):
        ax.text(v + 0.5, yv, f'{v:.0f}%', va='center', color='#52514e', fontsize=9)
plt.tight_layout(); plt.show()

# Do the data-driven clusters agree with the rule-based RFM segments?
ct = pd.crosstab(rfm['Segment'], rfm['RFM_Segment'], normalize='index').reindex(seg_list)
plt.figure(figsize=(12, 3.8))
sns.heatmap(ct, annot=True, fmt='.0%', cmap='Blues', cbar=False)
plt.title('Each K-Means segment broken down by rule-based RFM segment (rows sum to 100%)'); plt.ylabel(''); plt.xlabel('')
plt.show()

print('Share of DBSCAN outliers (whales) in each segment:')
print(rfm.assign(Whale=rfm['DBSCAN'] == -1).groupby('Segment')['Whale'].mean().reindex(seg_list).map('{:.1%}'.format))

**Interpretation.** The segments are named from their statistics (table above), not chosen in advance:

* **Champions:** most recent, most frequent, highest spend. A minority of customers carrying most of the revenue, and home to nearly all DBSCAN whales.
* **Loyal Customers** *(if present at your K)*: regular buyers above the median on all three dimensions but below the Champions.
* **At-Risk High-Value:** their past frequency and spend are above median, but recency is poor. They used to matter and have gone quiet. This revenue is at risk of being lost.
* **New Inflow:** bought recently but only once or twice. Their value is still undecided.
* **Hibernating / Lost:** old, infrequent, low spend. A large group by headcount and a small one by revenue.

The cross-tab against the rule-based RFM segments shows that the clusters largely **recover the same groups** the RFM rules define. Two independent methods agreeing makes the segmentation defensible.

## 14. Business Recommendations

In [ ]:
PLAYBOOK = {
    'Champions': ('Reward, don\'t discount', 'Loyalty perks, early access to new ranges, referral programme, key-account manager for whales',
                  'Retention rate, share of wallet'),
    'Loyal Customers': ('Grow basket', 'Cross-sell related catalogue ranges, volume-tier pricing, "complete the set" recommendations',
                        'AOV, products per order'),
    'At-Risk High-Value': ('Win back now', 'Personal outreach + direct win-back email, time-limited offer, ask why they left',
                           'Reactivation rate within 60 days'),
    'New Inflow': ('Trigger 2nd purchase', 'Welcome sequence, discount on second order, best-seller recommendations',
                   '2nd-order conversion within 90 days'),
    'Needs Attention': ('Re-engage', 'Personalised recommendations from past purchases, seasonal catalogue reminders',
                        'Order frequency'),
    'Hibernating / Lost': ('Low-cost reactivation', 'Automated low-cost campaigns only (Q4 catalogue); suppress from expensive channels',
                           'Cost per reactivated customer'),
}
def play(seg):
    key = next((k for k in PLAYBOOK if seg.startswith(k)), 'Needs Attention')
    return PLAYBOOK[key]

reco = prof.set_index('Segment')[['Customers', 'Customers_%', 'Revenue_%', 'Recency_med', 'Frequency_med', 'Monetary_med']].copy()
reco[['Objective', 'Action', 'KPI to track']] = [play(s) for s in reco.index]
reco

### Recommendations by area

**Customer segments** (table above)
1. **Protect Champions without discounting them.** They buy anyway. Use recognition and service (early access, priority stock, a named account manager for whale accounts flagged by DBSCAN) rather than price cuts.
2. **Spend the retention budget on At-Risk High-Value customers first.** They have the highest proven value per head and the clearest exit signal (falling R score). Contact them personally, before the Q4 season.
3. **Convert New Inflow before they go cold.** A large share of all customers never place a second order. An automated second-purchase journey within 30–60 days of the first order is the cheapest growth lever.
4. **Keep Hibernating / Lost on low-cost channels only**, and include them in the pre-Christmas catalogue send, when reactivation odds are highest.

**Operations & merchandising**
5. **Plan inventory and capacity for September–November**, and keep the Pareto-core SKUs at safety stock.
6. **Clear or delist dead stock** before the peak to free warehouse space.
7. **Freeze deployments in Q4; run maintenance on Saturdays / overnight** when traffic is zero.

**Commercial**
8. **Formalise a wholesale/trade programme** (tiered pricing, trade-pack sizes, account portal). The data already shows strong trade-buying behaviour, especially internationally.
9. **Grow international markets with many mid-size customers** (e.g. Germany, France) through localised marketing, and manage the few large distributors (e.g. the Netherlands, Australia) as key accounts.
10. **Reduce costly cancellations:** confirm very large orders before fulfilment and review high-return products and customers.
11. **Capture guest checkouts** (account creation incentives) so that revenue can be included in CRM and segmentation.

**Operationalise:** re-run this notebook monthly, store each customer's segment in the CRM, and track how many customers move *between* segments (e.g. Loyal → At-Risk) as an early churn KPI. Measure every campaign with a **hold-out control group**: this analysis is descriptive, so only experiments can prove that an action *causes* a change.

In [ ]:
# Export the customer-level results for CRM / marketing use
export_cols = ['Country', 'Recency', 'Frequency', 'Monetary', 'AOV', 'TenureDays', 'ReturnRate',
               'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Segment', 'KMeans', 'Segment', 'DBSCAN']
out = rfm[export_cols].assign(IsWhale=rfm['DBSCAN'] == -1).drop(columns='DBSCAN')
out.to_csv(f'{OUT_DIR}/customer_segments.csv')
reco.to_csv(f'{OUT_DIR}/segment_recommendations.csv')
print(f'Saved {len(out):,} customers to {OUT_DIR}/customer_segments.csv')
out.head()

## 15. Limitations & Next Steps

**Limitations**
* **Descriptive, not causal.** The analysis shows *what* customers did, not *why*. Statements such as "Q4 campaigns increase revenue" need controlled experiments.
* **About a quarter of lines have no CustomerID.** Guest buyers are in the sales analysis but not in the segmentation, so the segments describe registered customers only.
* **One year of data with a partial final month.** Seasonality is seen only once; we can't separate a seasonal effect from a one-off 2011 trend. Recency is measured relative to 09-Dec-2011.
* **Cancellations of purchases made before Dec-2010** cannot be matched to their originals. They still reduce net revenue, but their link to the original sale is unknown.
* **No cost or margin data.** Monetary is revenue, not profit. A high-revenue, high-return or heavily discounted customer may be less valuable than they appear.
* **Mixed B2B/B2C base.** One RFM model treats a wholesaler and a gift shopper on the same scale. Separate models per customer type may be fairer.
* **Quintile ties:** ranking before `qcut` means customers with the same Frequency can receive different F scores.
* **K-Means assumptions:** it assumes compact, round clusters; customer value is in reality a continuum, so boundaries between neighbouring segments are soft.

**Next steps**
1. **Customer Lifetime Value** with BG/NBD + Gamma-Gamma models to predict future value, not just past value.
2. **Churn prediction model** using the R-score trend and order gaps as early-warning features.
3. **Market-basket analysis** (association rules) to power the cross-sell recommendations for Loyal Customers.
4. **Separate B2B and B2C segmentation**, and a UK vs international model.
5. **A/B test** each segment playbook against a hold-out group.
6. **Productionise:** move cleaning functions into `/src`, split this notebook into `01_cleaning.ipynb`, `02_eda.ipynb`, `03_rfm.ipynb`, `04_clustering.ipynb`, keep data references in `/data` and the final PDF summary in `/reports`.

## 16. Final Conclusion

This project followed one line of reasoning from raw transactions to business actions:

1. **Trustworthy data first.** Types were fixed and duplicates removed. Guest checkouts were identified rather than deleted, and adjustments and non-product codes were isolated. Cancellations were kept as negative revenue, so every figure is **net of returns**, which removed the distortion from a few giant reversed orders.
2. **The business, before the algorithms.** EDA showed a highly seasonal (Sep–Nov), office-hours, mid-week business. Revenue is concentrated in a small share of products and customers, and there is clear wholesale behaviour, especially abroad.
3. **RFM turned transactions into customer behaviour.** Recency, Frequency and Monetary were scored, explained in real units, and mapped to the business translation matrix.
4. **Defensible segments.** Log-transformed and standardised RFM features were clustered with K-Means. K was chosen with the elbow, silhouette, Davies–Bouldin and a stability check. The result was confirmed by hierarchical clustering, and DBSCAN was used to flag whale accounts. Segments were named from their statistics and agree with the rule-based RFM groups.
5. **Specific actions.** Reward Champions without discounts, win back At-Risk High-Value customers before Q4, drive a second purchase from New Inflow, and keep Lost customers on low-cost channels. All of this is backed by inventory, timing, wholesale and returns recommendations, and measured with KPIs and hold-out tests.

The outcome is a reproducible, evidence-based segmentation that marketing can run monthly to decide **who to contact, with what offer, and when**.